# Day 3 Session 1 — RAG Exercises

In the demos you saw how embeddings encode semantic meaning into vectors, how ChromaDB indexes those vectors for fast retrieval, and how section-aware chunking preserves document structure. Now you will combine these pieces into production-ready components for a consulting knowledge platform.

**Exercises in this notebook:**
1. Build a SearchEngine Class (Embeddings + ChromaDB) — *Required*
2. Embedding Similarity Explorer — *Required*
3. RAG Pipeline with LLM Answer Generation — *Required*
4. Metadata-Filtered Search — *Optional (Intermediate)*
5. Chunking Strategy Comparison — *Optional (Intermediate)*
6. Multi-Index RAG — *Optional (Intermediate)*

In [ ]:
!pip install -q openai chromadb python-dotenv langchain langchain-openai langchain-text-splitters numpy

---
## Setup

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["LANGCHAIN_TRACING_V2"] = os.getenv("LANGCHAIN_TRACING_V2", "false")
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT", "mckinsey-genai-3day")

import openai
import chromadb
import numpy as np
from langchain_openai import ChatOpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter

print("All imports successful!")

---
## Recap

From the demos:
- **Embeddings** turn text into dense vectors where cosine similarity reflects semantic closeness (Demo 1)
- **ChromaDB** stores those vectors in an HNSW index so that `collection.query()` returns nearest neighbours in sub-millisecond time (Demo 2)
- **Section-aware chunking** splits on `## ` headers first, keeping each logical section intact for higher-quality retrieval (Demo 3)
- ChromaDB returns **distances** (not similarities): lower distance = more similar

---
## Exercise 1 (Easy): Build a SearchEngine Class (Embeddings + ChromaDB)

**Reference:** Demo 1 (embeddings) + Demo 2 (ChromaDB indexing and querying)

> **Hint:** ChromaDB stores documents as embeddings. Use `collection.add()` to index and `collection.query()` to search.

Build a `SearchEngine` class that an engineer can drop into any service. It should:
1. Accept a list of consulting knowledge documents in `__init__`
2. Embed all documents using OpenAI embeddings and index them in a ChromaDB collection
3. Provide a `search(query, k)` method that returns the top-k results with their distances

**Requirements:**
- Use `chromadb.Client()` and `get_or_create_collection` with `cosine` distance
- Embed documents via `openai.OpenAI().embeddings.create()`
- The `search` method must embed the query and call `collection.query()`
- Return a list of `(distance, document)` tuples sorted by distance (lowest first)

**Step-by-step TODOs are provided in the skeleton below.**

In [ ]:
# Exercise 1 -- Build a SearchEngine class backed by ChromaDB

class SearchEngine:
    def __init__(self, documents):
        """Embed all documents and index them in a ChromaDB collection."""
        self.client = openai.OpenAI()
        self.chroma = chromadb.Client()
        self.collection = self.chroma.get_or_create_collection(
            name="search_engine_kb",
            metadata={"hnsw:space": "cosine"}
        )

        # Step 1: Embed all documents (provided)
        emb_response = self.client.embeddings.create(
            model=os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small"),
            input=documents
        )
        embeddings = [item.embedding for item in emb_response.data]

        # Step 2: Add to ChromaDB (provided)
        self.collection.add(
            documents=documents,
            embeddings=embeddings,
            ids=[f"doc_{i}" for i in range(len(documents))]
        )
        self.doc_count = len(documents)
        print(f"Indexed {self.doc_count} documents.")

    def search(self, query, k=3):
        """Search for the top-k most similar documents.
        
        Returns:
            list of dicts with keys: document, distance
        """
        # TODO Step 1: Embed the query
        # Hint: Same API call as __init__, but with input=[query]
        # query_emb = self.client.embeddings.create(
        #     model=os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small"),
        #     input=[query]
        # ).data[0].embedding

        # TODO Step 2: Query ChromaDB
        # Hint: results = self.collection.query(query_embeddings=[query_emb], n_results=k)

        # TODO Step 3: Return results as list of dicts
        # Hint: return [{"document": doc, "distance": dist}
        #        for doc, dist in zip(results["documents"][0], results["distances"][0])]
        
        pass  # <-- remove and uncomment the code above

In [ ]:
# --------------- Test Exercise 1 ---------------
# Uncomment after implementing the SearchEngine class

corpus = [
    "Digital transformation in financial services requires a phased approach starting with core banking modernization.",
    "Post-merger integration success depends on Day-1 readiness and a structured 100-day plan for synergy capture.",
    "Omnichannel retail strategy must unify inventory, pricing, and customer data across physical and digital channels.",
    "Private equity value creation levers include revenue growth, operational efficiency, and strategic bolt-on acquisitions.",
    "Supply chain resilience requires diversification of sourcing, near-shoring critical components, and end-to-end visibility.",
    "Organizational health, as measured by the OHI, is the strongest predictor of sustained long-term performance.",
    "ESG strategy must be embedded into core business operations and capital allocation to create shareholder value.",
    "Cross-border M&A transactions require careful due diligence on regulatory, tax, and cultural integration risks."
]

# engine = SearchEngine(corpus)
# for query in [
#     "What are best practices for post-merger integration?",
#     "How should a retailer approach omnichannel transformation?",
#     "What are the key risks in cross-border M&A?"
# ]:
#     print(f"\nQuery: {query}")
#     results = engine.search(query, k=3)
#     for dist, doc in results:
#         print(f"  {dist:.4f} | {doc}")

### Expected Output (Exercise 1)

```
Indexed 8 documents in search_engine_kb

Query: What are best practices for post-merger integration?
  0.3842 | Post-merger integration success depends on Day-1 readiness and a structured 100-day plan for synergy capture.
  0.5765 | Cross-border M&A transactions require careful due diligence on regulatory, tax, and cultural integration risks.
  0.6700 | Private equity value creation levers include revenue growth, operational efficiency, and strategic bolt-on acquisitions.

Query: How should a retailer approach omnichannel transformation?
  0.3535 | Omnichannel retail strategy must unify inventory, pricing, and customer data across physical and digital channels.
  0.4778 | Digital transformation in financial services requires a phased approach starting with core banking modernization.
  0.5739 | Organizational health, as measured by the OHI, is the strongest predictor of sustained long-term performance.

Query: What are the key risks in cross-border M&A?
  0.2713 | Cross-border M&A transactions require careful due diligence on regulatory, tax, and cultural integration risks.
  0.6283 | Post-merger integration success depends on Day-1 readiness and a structured 100-day plan for synergy capture.
  0.6883 | Supply chain resilience requires diversification of sourcing, near-shoring critical components, and end-to-end visibility.
```

*Exact distances will vary slightly between runs but the ranking order should be stable. Lower distance = more similar (cosine distance, not cosine similarity).*

### Progressive Hints (Exercise 1)

<details>
<summary>Hint 1: Embedding API call</summary>

```python
emb_response = self.client.embeddings.create(
    model=os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small"),
    input=documents
)
```
</details>

<details>
<summary>Hint 2: Adding to ChromaDB</summary>

```python
self.collection.add(
    documents=documents,
    embeddings=doc_embeddings,
    ids=[f"doc_{i}" for i in range(len(documents))]
)
```
</details>

<details>
<summary>Hint 3: Complete search method</summary>

```python
def search(self, query, k=3):
    response = self.client.embeddings.create(
        model=os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small"),
        input=[query]
    )
    query_emb = response.data[0].embedding
    results = self.collection.query(query_embeddings=[query_emb], n_results=k)
    return list(zip(results['distances'][0], results['documents'][0]))
```
</details>

---
## Exercise 2 (Easy): Embedding Similarity Explorer

**Reference:** Demo 1 (embeddings, cosine similarity, similarity matrix)

> **Hint:** The RAG pattern is: embed query -> search vector store -> format context -> send to LLM with context + question.

Build utility functions that help engineers understand the semantic relationships in their document corpus. This is essential for debugging retrieval quality — if your embeddings don't capture the right relationships, your RAG system will retrieve irrelevant content.

**You will implement three functions:**
1. `embed_texts(texts)` — Embed a list of texts and return the vectors
2. `find_most_and_least_similar(texts, embeddings)` — Find the most and least similar pair
3. `detect_outliers(texts, embeddings, threshold=0.15)` — Find texts with low average similarity to all others (potential outliers or miscategorized documents)

In [ ]:
# Exercise 2 -- Embedding Similarity Explorer

def cosine_similarity(a, b):
    """Compute cosine similarity between two vectors."""
    a, b = np.array(a), np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


def embed_texts(texts):
    """Embed a list of texts using OpenAI's embedding model. (Provided)"""
    client = openai.OpenAI()
    response = client.embeddings.create(
        model=os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small"),
        input=texts
    )
    return [item.embedding for item in response.data]


def find_most_similar(query, texts, embeddings):
    """
    Find the most similar text to a query.
    (Provided)
    """
    query_emb = embed_texts([query])[0]
    scores = [cosine_similarity(query_emb, emb) for emb in embeddings]
    best_idx = int(np.argmax(scores))
    return {
        "most_similar_text": texts[best_idx],
        "similarity_score": scores[best_idx],
        "all_scores": list(zip(texts, scores))
    }


def build_similarity_matrix(texts, embeddings):
    """
    Build an NxN similarity matrix between all texts.
    
    TODO: Fill in the nested loop to compute pairwise cosine similarity.
    """
    n = len(texts)
    matrix = np.zeros((n, n))
    
    # TODO: Fill the matrix with pairwise cosine similarities
    # Hint: for i in range(n):
    #           for j in range(n):
    #               matrix[i][j] = cosine_similarity(embeddings[i], embeddings[j])
    
    return matrix

In [ ]:
# --------------- Test Exercise 2 ---------------
# Uncomment after implementing the functions

# test_texts = [
#     "Digital transformation in financial services requires a phased approach.",
#     "Post-merger integration success depends on Day-1 readiness.",
#     "Omnichannel retail strategy must prioritize seamless customer experience.",
#     "Supply chain resilience requires diversification of sourcing.",
#     "The weather today is sunny and warm with clear skies.",  # Outlier!
#     "A recipe for chocolate cake requires flour, sugar, and cocoa."  # Outlier!
# ]
# 
# # Step 1: Embed
# test_embeddings = embed_texts(test_texts)
# print(f"Embedded {len(test_embeddings)} texts, each with {len(test_embeddings[0])} dimensions")
# 
# # Step 2: Find most/least similar
# most, least = find_most_and_least_similar(test_texts, test_embeddings)
# print(f"\nMost similar pair (score={most[0]:.4f}):")
# print(f"  A: {most[1][:70]}")
# print(f"  B: {most[2][:70]}")
# print(f"\nLeast similar pair (score={least[0]:.4f}):")
# print(f"  A: {least[1][:70]}")
# print(f"  B: {least[2][:70]}")
# 
# # Step 3: Detect outliers
# outliers = detect_outliers(test_texts, test_embeddings, threshold=0.15)
# print(f"\nOutliers (avg similarity < 0.15):")
# for avg_sim, text in outliers:
#     print(f"  {avg_sim:.4f} | {text}")

### Expected Output (Exercise 2)

```
Embedded 6 texts, each with 1536 dimensions

Most similar pair (score=0.37xx):
  A: Digital transformation in financial services requires a phased appro...
  B: Omnichannel retail strategy must prioritize seamless customer experie...

Least similar pair (score=-0.05xx):
  A: Supply chain resilience requires diversification of sourcing.
  B: The weather today is sunny and warm with clear skies.

Outliers (avg similarity < 0.15):
  0.0xxx | The weather today is sunny and warm with clear skies.
  0.0xxx | A recipe for chocolate cake requires flour, sugar, and cocoa.
```

*The weather and cake texts should be detected as outliers since they have very low average similarity to the consulting corpus.*

### Progressive Hints (Exercise 2)

<details>
<summary>Hint 1: embed_texts function</summary>

```python
def embed_texts(texts):
    client = openai.OpenAI()
    response = client.embeddings.create(
        model=os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small"),
        input=texts
    )
    return [item.embedding for item in response.data]
```
</details>

<details>
<summary>Hint 2: Iterating over unique pairs</summary>

```python
for i in range(len(texts)):
    for j in range(i + 1, len(texts)):
        sim = cosine_similarity(embeddings[i], embeddings[j])
        # compare with current best/worst
```
</details>

<details>
<summary>Hint 3: Computing average similarity for outlier detection</summary>

```python
for i in range(len(texts)):
    sims = [cosine_similarity(embeddings[i], embeddings[j])
            for j in range(len(texts)) if j != i]
    avg_sim = np.mean(sims)
```
</details>

---
## Exercise 3 (Easy): RAG Pipeline with LLM Answer Generation

**Reference:** Demo 2 (ChromaDB retrieval) + Day 2 Session 2 Demo 5 (RAG pattern with LLM)

> **Hint:** Each chain step feeds its output as input to the next. Build and test each step independently first.

Extend the `SearchEngine` from Exercise 1 with an `ask(question, k=3)` method that:
1. Retrieves the top-k most relevant chunks from ChromaDB
2. Formats them as context for the LLM
3. Sends the question + context to GPT to generate a cited answer
4. Returns the answer with source references

This is the complete RAG pipeline: **Retrieve -> Augment -> Generate**.

In [ ]:
# Exercise 3 -- RAG Pipeline with LLM Answer Generation

class RAGSearchEngine:
    def __init__(self, documents):
        """Initialize: embed documents and index in ChromaDB. (Provided)"""
        self.client = openai.OpenAI()
        self.chroma = chromadb.Client()
        self.collection = self.chroma.get_or_create_collection(
            name="rag_engine_kb",
            metadata={"hnsw:space": "cosine"}
        )
        self.documents = documents

        # Embed and index (same as Exercise 1)
        emb_response = self.client.embeddings.create(
            model=os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small"),
            input=documents
        )
        embeddings = [item.embedding for item in emb_response.data]
        self.collection.add(
            documents=documents,
            embeddings=embeddings,
            ids=[f"doc_{i}" for i in range(len(documents))]
        )
        print(f"RAG engine indexed {len(documents)} documents.")

    def search(self, query, k=3):
        """Search for top-k documents. (Provided)"""
        query_emb = self.client.embeddings.create(
            model=os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small"),
            input=[query]
        ).data[0].embedding
        results = self.collection.query(query_embeddings=[query_emb], n_results=k)
        return results["documents"][0]

    def ask(self, question, k=3):
        """
        Full RAG pipeline: retrieve context, then generate an answer.
        
        TODO: Implement this method.
        Steps:
          1. Call self.search(question, k) to get relevant chunks
          2. Format them as numbered context: "[1] chunk1\n[2] chunk2\n..."
          3. Call self.client.chat.completions.create() with:
             - system message: "Answer using ONLY the context provided. Cite sources as [1], [2], etc."
             - user message: context + question
          4. Return the answer text
        """
        # Hint:
        # chunks = self.search(question, k)
        # context = "\n".join([f"[{i+1}] {c}" for i, c in enumerate(chunks)])
        # response = self.client.chat.completions.create(
        #     model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
        #     messages=[
        #         {"role": "system", "content": "Answer the question using ONLY the provided context. Cite sources as [1], [2], etc.\n\nContext:\n" + context},
        #         {"role": "user", "content": question}
        #     ],
        #     max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "300"))
        # )
        # return response.choices[0].message.content
        pass  # <-- remove and uncomment above

In [ ]:
# --------------- Test Exercise 3 ---------------
# Uncomment after implementing RAGSearchEngine

# corpus = [
#     "Digital transformation in financial services requires a phased approach starting with core banking modernization.",
#     "Post-merger integration success depends on Day-1 readiness and a structured 100-day plan for synergy capture.",
#     "Omnichannel retail strategy must unify inventory, pricing, and customer data across physical and digital channels.",
#     "Private equity value creation levers include revenue growth, operational efficiency, and strategic bolt-on acquisitions.",
#     "Supply chain resilience requires diversification of sourcing, near-shoring critical components, and end-to-end visibility.",
#     "Organizational health, as measured by the OHI, is the strongest predictor of sustained long-term performance.",
#     "ESG strategy must be embedded into core business operations and capital allocation to create shareholder value.",
#     "Cross-border M&A transactions require careful due diligence on regulatory, tax, and cultural integration risks."
# ]
# 
# rag_engine = RAGSearchEngine(corpus)
# 
# # Test with consulting questions
# questions = [
#     "What should a company focus on in the first 100 days after a merger?",
#     "How can a private equity firm create value in a portfolio company?",
#     "What is the recipe for chocolate cake?"  # Should say insufficient info
# ]
# 
# for q in questions:
#     print(f"\n{'='*60}")
#     print(f"Question: {q}")
#     result = rag_engine.ask(q, k=3)
#     print(f"\nAnswer: {result['answer']}")
#     print(f"\nSources used:")
#     for i, src in enumerate(result['sources']):
#         print(f"  [{i+1}] {src[:80]}...")

### Expected Output (Exercise 3)

```
============================================================
Question: What should a company focus on in the first 100 days after a merger?

Answer: Based on the provided context, a company should focus on Day-1 readiness 
and follow a structured 100-day plan for synergy capture [Source 1]. The integration 
should also include careful due diligence on regulatory, tax, and cultural integration 
risks [Source 2].

Sources used:
  [1] Post-merger integration success depends on Day-1 readiness and a structur...
  [2] Cross-border M&A transactions require careful due diligence on regulatory...
  [3] Private equity value creation levers include revenue growth, operational e...

============================================================
Question: How can a private equity firm create value in a portfolio company?

Answer: According to the context, private equity value creation levers include 
revenue growth, operational efficiency, and strategic bolt-on acquisitions [Source 1].

Sources used:
  [1] Private equity value creation levers include revenue growth, operational e...
  [2] Digital transformation in financial services requires a phased approach sta...
  [3] Organizational health, as measured by the OHI, is the strongest predictor ...

============================================================
Question: What is the recipe for chocolate cake?

Answer: I don't have enough information in the provided context to answer this question. 
The available sources cover consulting topics like M&A, strategy, and operations, 
but do not contain any information about recipes or cooking.

Sources used:
  [1] ...
```

*The exact wording will vary by LLM run, but the structure (cited answer + sources) should be consistent.*

### Progressive Hints (Exercise 3)

<details>
<summary>Hint 1: The system prompt</summary>

```python
system_prompt = """You are a McKinsey knowledge assistant. Answer the user's question 
based ONLY on the provided context. Cite your sources using [Source N] notation. 
If the context does not contain enough information to answer the question, 
say "I don't have enough information in the provided context to answer this question.""""
```
</details>

<details>
<summary>Hint 2: Calling the LLM</summary>

```python
response = self.client.chat.completions.create(
    model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"}
    ],
    temperature=0.2
)
answer = response.choices[0].message.content
```
</details>

<details>
<summary>Hint 3: Complete ask method</summary>

```python
def ask(self, question, k=3):
    results = self.search(question, k)
    context_parts = [f"[Source {i+1}] {doc}" for i, (dist, doc) in enumerate(results)]
    context = "\n".join(context_parts)
    
    system_prompt = """You are a McKinsey knowledge assistant. Answer based ONLY on the provided context. Cite sources using [Source N]. If insufficient info, say so."""
    
    response = self.client.chat.completions.create(
        model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"}
        ],
        temperature=0.2
    )
    
    return {
        "answer": response.choices[0].message.content,
        "sources": [doc for (dist, doc) in results],
        "context_used": context
    }
```
</details>

---
---
# Optional Exercises (Intermediate)


> **Hint:** Review the referenced demos for the exact patterns to follow. The optional exercises combine multiple demo patterns.
These exercises go deeper into production RAG concerns. Complete them if you finish Exercises 1-3 early or want extra practice.

---
## Optional Exercise 1 (Intermediate): Metadata-Filtered Search

**Reference:** Demo 2 (ChromaDB with metadata)


> **Hint:** Review the referenced demos for the exact patterns to follow. The optional exercises combine multiple demo patterns.
In production, consultants often want to search within a specific practice area or industry. Add metadata (practice_area, industry) to your documents and implement filtered search.

**Requirements:**
- Modify SearchEngine to accept `metadatas` parameter in `__init__`
- Add a `filtered_search(query, k, where)` method that uses ChromaDB's `where` clause
- Test with queries filtered by practice_area and industry

In [ ]:
# Optional Exercise 1 -- Metadata-Filtered Search

class MetadataSearchEngine:
    def __init__(self, documents, metadatas=None):
        """
        Initialize with documents and optional metadata for each document.
        
        Args:
            documents: List of text strings
            metadatas: List of dicts with keys like 'practice_area', 'industry'
        """
        self.client = openai.OpenAI()
        self.chroma = chromadb.Client()
        self.collection = self.chroma.get_or_create_collection(
            name="metadata_search_kb",
            metadata={"hnsw:space": "cosine"}
        )

        # TODO: Embed documents (same as Exercise 1)
        
        # TODO: Add documents with metadatas parameter
        #   self.collection.add(
        #       documents=documents,
        #       embeddings=doc_embeddings,
        #       ids=[f"doc_{i}" for i in range(len(documents))],
        #       metadatas=metadatas  # Pass the metadata dicts
        #   )
        
        pass

    def search(self, query, k=3):
        """Unfiltered search (same as Exercise 1)."""
        # TODO: Implement (same as Exercise 1)
        pass

    def filtered_search(self, query, k=3, where=None):
        """
        Search with metadata filtering.
        
        Args:
            query: Search query string
            k: Number of results
            where: ChromaDB where clause, e.g. {"practice_area": "M&A"}
                   or {"$and": [{"practice_area": "M&A"}, {"industry": "Healthcare"}]}
        
        Returns:
            List of (distance, document, metadata) tuples
        """
        # TODO Step 1: Embed the query
        
        # TODO Step 2: Call collection.query with where parameter
        #   results = self.collection.query(
        #       query_embeddings=[query_emb],
        #       n_results=k,
        #       where=where  # This filters by metadata!
        #   )
        
        # TODO Step 3: Return (distance, document, metadata) tuples
        #   return list(zip(
        #       results['distances'][0],
        #       results['documents'][0],
        #       results['metadatas'][0]
        #   ))
        
        pass

In [ ]:
# --------------- Test Optional Exercise 1 ---------------
# Uncomment after implementing MetadataSearchEngine

# documents = [
#     "Digital transformation in financial services requires a phased approach.",
#     "Post-merger integration should follow a 100-day plan.",
#     "Omnichannel retail strategy must unify inventory and pricing.",
#     "Private equity value creation levers include revenue growth.",
#     "Healthcare cost transformation requires addressing clinical variation.",
#     "Cross-border M&A transactions require careful due diligence.",
#     "ESG strategy should be embedded into core business operations.",
#     "Pharmaceutical supply chain needs cold-chain logistics optimization."
# ]
# 
# metadatas = [
#     {"practice_area": "Digital", "industry": "Financial Services"},
#     {"practice_area": "M&A", "industry": "Cross-Industry"},
#     {"practice_area": "Retail", "industry": "Retail"},
#     {"practice_area": "Private Equity", "industry": "Cross-Industry"},
#     {"practice_area": "Healthcare", "industry": "Healthcare"},
#     {"practice_area": "M&A", "industry": "Cross-Industry"},
#     {"practice_area": "Sustainability", "industry": "Cross-Industry"},
#     {"practice_area": "Operations", "industry": "Healthcare"},
# ]
# 
# engine = MetadataSearchEngine(documents, metadatas)
# 
# query = "What are the key considerations for a merger?"
# 
# print("Unfiltered search:")
# for dist, doc in engine.search(query, k=3):
#     print(f"  {dist:.4f} | {doc[:70]}")
# 
# print("\nFiltered to M&A practice area:")
# for dist, doc, meta in engine.filtered_search(query, k=3, where={"practice_area": "M&A"}):
#     print(f"  {dist:.4f} | [{meta['practice_area']}] {doc[:60]}")
# 
# print("\nFiltered to Healthcare industry:")
# for dist, doc, meta in engine.filtered_search(query, k=3, where={"industry": "Healthcare"}):
#     print(f"  {dist:.4f} | [{meta['industry']}] {doc[:60]}")

---
## Optional Exercise 2 (Intermediate): Chunking Strategy Comparison

**Reference:** Demo 3 (section-aware vs naive chunking)


> **Hint:** Review the referenced demos for the exact patterns to follow. The optional exercises combine multiple demo patterns.
Test different chunk_size/overlap combinations and measure retrieval quality. The goal is to find the configuration that consistently retrieves the correct document chunk for a set of test queries.

**Requirements:**
- Define a test document (use the PMI report from Demo 3)
- Define a set of test queries with known "correct" chunks
- Test 3 different (chunk_size, chunk_overlap) combinations
- Measure: Does the correct chunk appear in top-3 retrieved results?
- Report accuracy for each configuration

In [ ]:
# Optional Exercise 2 -- Chunking Strategy Comparison

from langchain_text_splitters import RecursiveCharacterTextSplitter

# The same PMI report from Demo 3
sample_doc = """# Post-Merger Integration: A Best Practice Guide

Post-merger integration (PMI) is the critical phase that determines whether an M&A transaction delivers its intended value. McKinsey research shows that 70% of mergers fail to achieve projected synergies, most often due to integration execution failures.

## Executive Summary

Successful post-merger integration requires a structured 100-day plan, early synergy identification, cultural alignment, and disciplined execution. Our analysis of 200+ transactions reveals that Day-1 readiness is the single strongest predictor of integration success.

## Key Findings

Three factors distinguish successful integrations from unsuccessful ones:

1. Leadership alignment: Joint leadership teams must be appointed within the first two weeks, with clear decision rights and accountability.
2. Synergy capture: Revenue and cost synergies must be quantified with bottom-up rigor and tracked through a dedicated Integration Management Office (IMO).
3. Cultural integration: Cultural differences must be proactively addressed through joint team workshops, shared values articulation, and visible leadership behavior.

## Recommendations

We recommend a phased approach to integration. Phase 1 (Days 1-30) focuses on stabilization and quick wins. Phase 2 (Days 31-100) addresses operating model design and synergy capture. Phase 3 (Months 4-12) drives full integration and transformation.

## Appendix

Detailed case studies from recent financial services and healthcare M&A transactions are provided in the supplementary materials, including synergy tracking templates and cultural assessment frameworks."""

# Test queries with expected content (substring that should appear in the correct chunk)
test_queries = [
    {
        "query": "What is the strongest predictor of integration success?",
        "expected_substring": "Day-1 readiness is the single strongest predictor"
    },
    {
        "query": "How should leadership teams be structured after a merger?",
        "expected_substring": "Joint leadership teams must be appointed"
    },
    {
        "query": "What happens in Phase 2 of integration?",
        "expected_substring": "Phase 2 (Days 31-100) addresses operating model design"
    },
    {
        "query": "What percentage of mergers fail?",
        "expected_substring": "70% of mergers fail"
    }
]

# Chunking configurations to test
configs = [
    {"chunk_size": 150, "chunk_overlap": 20, "label": "Small (150/20)"},
    {"chunk_size": 300, "chunk_overlap": 50, "label": "Medium (300/50)"},
    {"chunk_size": 500, "chunk_overlap": 80, "label": "Large (500/80)"},
]

def evaluate_chunking_config(doc, config, queries, k=3):
    """
    Evaluate a chunking configuration by measuring retrieval accuracy.
    
    Args:
        doc: The full document text
        config: Dict with chunk_size, chunk_overlap, label
        queries: List of dicts with 'query' and 'expected_substring'
        k: Number of results to retrieve
    
    Returns:
        accuracy (float): Fraction of queries where correct chunk is in top-k
    """
    # TODO Step 1: Create a RecursiveCharacterTextSplitter with the config
    #   splitter = RecursiveCharacterTextSplitter(
    #       chunk_size=config["chunk_size"],
    #       chunk_overlap=config["chunk_overlap"],
    #       separators=["\n## ", "\n# ", "\n\n", "\n", ". ", " "]
    #   )
    
    # TODO Step 2: Split the document into chunks
    #   chunks = splitter.split_text(doc)
    
    # TODO Step 3: Embed all chunks
    #   client = openai.OpenAI()
    #   chunk_embeddings = [item.embedding for item in
    #       client.embeddings.create(model=os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small"), input=chunks).data]
    
    # TODO Step 4: For each test query:
    #   a) Embed the query
    #   b) Compute cosine similarity with all chunks
    #   c) Get top-k chunks
    #   d) Check if any top-k chunk contains the expected_substring
    #   e) Count hits
    
    # TODO Step 5: Return accuracy = hits / len(queries)
    
    pass


# --------------- Run evaluation ---------------
# Uncomment after implementing evaluate_chunking_config

# print("Chunking Strategy Comparison")
# print("=" * 50)
# for config in configs:
#     accuracy = evaluate_chunking_config(sample_doc, config, test_queries, k=3)
#     print(f"  {config['label']:20s} -> Accuracy: {accuracy:.0%} ({int(accuracy * len(test_queries))}/{len(test_queries)} queries)")
# print("\nBest config has highest accuracy (correct chunk in top-3).")

---
## Optional Exercise 3 (Intermediate): Multi-Index RAG

**Reference:** Combines all 3 demos (embeddings, vector stores, chunking)


> **Hint:** Review the referenced demos for the exact patterns to follow. The optional exercises combine multiple demo patterns.
Build a system with:
1. **Two separate ChromaDB collections** — one for strategy documents, one for operations documents
2. **A router** that decides which collection to search based on the query
3. **A synthesis step** that combines results from both collections when needed

This pattern is common in enterprise RAG where different knowledge domains have different retrieval needs.

In [ ]:
# Optional Exercise 3 -- Multi-Index RAG

class MultiIndexRAG:
    def __init__(self):
        """
        Initialize two separate ChromaDB collections:
        - strategy_collection: Strategy, M&A, and PE documents
        - operations_collection: Operations, supply chain, and healthcare documents
        """
        self.client = openai.OpenAI()
        self.chroma = chromadb.Client()
        
        # Strategy documents
        self.strategy_docs = [
            "Digital transformation in financial services requires digitizing core processes, building new digital offerings, and reimagining the business model.",
            "Post-merger integration should follow a 100-day plan covering Day-1 readiness, synergy capture, cultural integration, and operating model design.",
            "Private equity value creation levers include revenue growth acceleration, operational efficiency, strategic M&A, and balance sheet optimization.",
            "Growth strategy in emerging markets requires local partnerships, adapted product offerings, and phased market entry.",
            "Corporate strategy must balance portfolio optimization, organic growth investments, and inorganic expansion through M&A."
        ]
        
        # Operations documents
        self.operations_docs = [
            "Supply chain resilience requires diversification of sourcing, near-shoring critical components, and end-to-end visibility.",
            "Lean manufacturing principles include waste elimination, continuous improvement, just-in-time inventory, and total quality management.",
            "Healthcare cost transformation requires addressing clinical variation, supply chain optimization, and administrative simplification.",
            "Procurement optimization should target spend analytics, supplier consolidation, demand management, and contract renegotiation.",
            "Warehouse automation with robotics and AI-driven inventory management can reduce fulfillment costs by 30-40%."
        ]
        
        # TODO Step 1: Create two ChromaDB collections
        #   self.strategy_collection = self.chroma.get_or_create_collection(
        #       name="strategy_kb", metadata={"hnsw:space": "cosine"}
        #   )
        #   self.operations_collection = self.chroma.get_or_create_collection(
        #       name="operations_kb", metadata={"hnsw:space": "cosine"}
        #   )
        
        # TODO Step 2: Embed and index strategy documents into strategy_collection
        
        # TODO Step 3: Embed and index operations documents into operations_collection
        
        pass

    def route_query(self, query):
        """
        Decide which collection(s) to search based on the query.
        
        Returns:
            'strategy', 'operations', or 'both'
        """
        # TODO: Use the LLM to classify the query
        #   Option A (simple): Keyword-based routing
        #     - If query mentions 'M&A', 'merger', 'strategy', 'growth', 'PE' -> 'strategy'
        #     - If query mentions 'supply chain', 'operations', 'manufacturing', 'cost' -> 'operations'
        #     - Otherwise -> 'both'
        #
        #   Option B (advanced): Use the LLM to classify
        #     response = self.client.chat.completions.create(
        #         model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
        #         messages=[{"role": "system", "content": "Classify this query..."},
        #                   {"role": "user", "content": query}],
        #         temperature=0
        #     )
        #     return response.choices[0].message.content.strip().lower()
        
        pass

    def search_collection(self, collection, query, k=3):
        """
        Search a specific collection.
        Returns list of (distance, document) tuples.
        """
        # TODO: Embed query and search the given collection
        
        pass

    def ask(self, question, k=3):
        """
        Full multi-index RAG pipeline:
        1. Route the query to the appropriate collection(s)
        2. Retrieve from selected collection(s)
        3. Synthesize an answer with citations
        
        Returns:
            dict with 'answer', 'route', 'sources'
        """
        # TODO Step 1: Route the query
        #   route = self.route_query(question)
        
        # TODO Step 2: Search the appropriate collection(s)
        #   if route == 'strategy':
        #       results = self.search_collection(self.strategy_collection, question, k)
        #   elif route == 'operations':
        #       results = self.search_collection(self.operations_collection, question, k)
        #   else:  # 'both'
        #       strategy_results = self.search_collection(self.strategy_collection, question, k)
        #       ops_results = self.search_collection(self.operations_collection, question, k)
        #       # Merge and sort by distance
        #       results = sorted(strategy_results + ops_results, key=lambda x: x[0])[:k]
        
        # TODO Step 3: Generate answer with LLM (same pattern as Exercise 3)
        #   Format context, call LLM, return structured result
        
        # TODO Step 4: Return dict with 'answer', 'route', 'sources'
        
        pass

In [ ]:
# --------------- Test Optional Exercise 3 ---------------
# Uncomment after implementing MultiIndexRAG

# rag = MultiIndexRAG()
# 
# test_questions = [
#     "What are the key levers for PE value creation?",          # Should route to strategy
#     "How can we reduce supply chain costs?",                    # Should route to operations
#     "How does digital transformation affect manufacturing?",   # Should route to both
# ]
# 
# for q in test_questions:
#     print(f"\n{'='*60}")
#     print(f"Question: {q}")
#     result = rag.ask(q, k=3)
#     print(f"Route: {result['route']}")
#     print(f"Answer: {result['answer'][:200]}...")
#     print(f"Sources ({len(result['sources'])}):")
#     for i, src in enumerate(result['sources']):
#         print(f"  [{i+1}] {src[:70]}...")

### Hints (Optional Exercise 3)

<details>
<summary>Hint 1: Simple keyword-based router</summary>

```python
def route_query(self, query):
    query_lower = query.lower()
    strategy_keywords = ['m&a', 'merger', 'strategy', 'growth', 'pe', 'private equity', 'acquisition', 'portfolio']
    ops_keywords = ['supply chain', 'operations', 'manufacturing', 'cost', 'procurement', 'warehouse', 'lean']
    
    has_strategy = any(kw in query_lower for kw in strategy_keywords)
    has_ops = any(kw in query_lower for kw in ops_keywords)
    
    if has_strategy and not has_ops:
        return 'strategy'
    elif has_ops and not has_strategy:
        return 'operations'
    else:
        return 'both'
```
</details>

<details>
<summary>Hint 2: Merging results from both collections</summary>

```python
# Search both collections
strategy_results = self.search_collection(self.strategy_collection, question, k)
ops_results = self.search_collection(self.operations_collection, question, k)

# Merge and take top-k by distance
all_results = strategy_results + ops_results
all_results.sort(key=lambda x: x[0])  # Sort by distance (lowest first)
results = all_results[:k]
```
</details>

---
## Wrap-Up

**What you built in this session:**
- Exercise 1: A reusable `SearchEngine` class combining embeddings + ChromaDB
- Exercise 2: Diagnostic tools for understanding embedding quality
- Exercise 3: A complete RAG pipeline with LLM answer generation and citations

**Optional exercises explored:**
- Metadata filtering for domain-specific retrieval
- Chunking optimization through empirical evaluation
- Multi-index architectures for enterprise knowledge bases

**Next:** In Session 2 (Deployment) you will deploy these components as APIs. In Session 3 (Capstone), you will combine everything into a production-grade consulting knowledge system.